# BIST-Analyst: fine-tuning Qwen3-8B on Kaggle

**Settings (right panel):** Accelerator = **GPU T4 x2** (not P100: the `--liger` kernels need compute capability ≥ 7.0),
Internet = **On**, Persistence = *Files only* (keeps `/kaggle/working` between sessions).

**Long run:** use **Save Version → Save & Run All (Commit)**: it runs in the background for up to 12 hours and keeps
`/kaggle/working` as the version output. Qwen3-8B on T4 ≈ 3 h per epoch.

Chat format: Qwen3 is rendered by `training/prompts.py` (every assistant turn starts with the empty
`<think>\n\n</think>\n\n` block, exactly like generation with `enable_thinking=False`), so training and inference match.

## 1. Code and dependencies

In [ ]:
%cd /kaggle/working
!rm -rf repo && git clone -q https://github.com/yarenerisen/bist100-agentic-financial-harness.git repo
%cd /kaggle/working/repo
!git checkout -q feature/tools-dataset-harness 2>/dev/null && echo "on feature branch" || echo "on main"
%cd /kaggle/working/repo/bist_llm
!pip install -q -U "transformers>=4.45" "peft>=0.13" "bitsandbytes>=0.44" accelerate pyarrow liger-kernel
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Settings

In [ ]:
MODEL   = 'Qwen/Qwen3-8B'
ADAPTER = '/kaggle/working/qwen3-8b-lora'      # adapter + checkpoints (kept in the version output)
RESULTS = '/kaggle/working/results'
EPOCHS  = 1          # eval loss plateaued after ~1 epoch in the 3B run; use 2 if time allows

## 3. Baseline: base model, full harness (for base vs fine-tuned)

In [ ]:
!python -m harness.evaluate --backend hf --model {MODEL} --load-4bit --limit 30 --mode E --tag base_E --out-dir {RESULTS}

## 4. Smoke test (30 examples) — checks memory and the Qwen3 format before the long run

In [ ]:
!python training/finetune_qlora.py --model {MODEL} --out /kaggle/working/smoke --limit 30 --epochs 1 --save-steps 1000 --liger

## 5. Full training (checkpoint every 20 steps; re-running this cell resumes)

In [ ]:
!python training/finetune_qlora.py --model {MODEL} --out {ADAPTER} --epochs {EPOCHS} --liger --resume
!cat {ADAPTER}/training_report.json | head -c 600

## 6. Evaluate the fine-tuned model (adapter loaded directly, 4-bit)

In [ ]:
!python -m harness.evaluate --backend hf --model {ADAPTER} --load-4bit --mode E --tag finetuned_E --out-dir {RESULTS}
!python -m harness.evaluate --backend hf --model {ADAPTER} --load-4bit --limit 15 --repeats 3 --tag finetuned_repro --out-dir {RESULTS}

## 7. Harness ablation A–E (same model, different harness levels)

In [ ]:
for m in 'ABCD':
    !python -m harness.evaluate --backend hf --model {ADAPTER} --load-4bit --limit 20 --mode {m} --tag finetuned_{m} --out-dir {RESULTS}
for m in 'AB':
    !python -m harness.evaluate --backend hf --model {MODEL} --load-4bit --limit 20 --mode {m} --tag base_{m} --out-dir {RESULTS}

## 8. Results table

In [ ]:
import json, glob, pandas as pd
rows = [{'run': json.load(open(f))['tag'], **json.load(open(f))['overall']} for f in sorted(glob.glob(f'{RESULTS}/eval_*.json'))]
pd.DataFrame(rows).set_index('run')

## 9. Export to GGUF for local, offline inference (llama.cpp / Ollama)
Merging needs ~16 GB RAM for 8B (Kaggle has ~29 GB). The Q4_K_M file (~5 GB) runs on an 8 GB laptop GPU.
Use the **raw** backends so the prompt format is identical to training:
`python -m harness.run_agent --backend llamacpp "..."` or `--backend ollama --model bist-analyst "..."`.

In [ ]:
!python training/finetune_qlora.py --merge --model {MODEL} --out {ADAPTER} --merged-out /tmp/merged
!git clone -q https://github.com/ggerganov/llama.cpp /tmp/llama.cpp && pip install -q -r /tmp/llama.cpp/requirements.txt
!python /tmp/llama.cpp/convert_hf_to_gguf.py /tmp/merged --outfile /tmp/bist-f16.gguf --outtype f16
!cd /tmp/llama.cpp && cmake -B build -DGGML_CUDA=OFF > /dev/null && cmake --build build --target llama-quantize -j 4 > /dev/null
!/tmp/llama.cpp/build/bin/llama-quantize /tmp/bist-f16.gguf /kaggle/working/bist-analyst-qwen3-8b-Q4_K_M.gguf Q4_K_M
!ls -lh /kaggle/working/*.gguf